# 01 — QoL_Stress dataset audit

Purpose: inspect the supplied CSV collections, participant linkage, labels, missingness and file integrity before selecting any modelling target. This notebook does not train a model and does not modify raw data.

Run from the `ml-workstream-2` project root. Dataset directory name is intentionally kept unchanged.

In [1]:
from pathlib import Path

# Notebook is inside ml-workstream-2/notebooks/
PROJECT_ROOT = Path.cwd().parent

DATASET_NAME = (
    "qol_stress-a-multimodal-dataset-of-physiological-and-self-reported-stress-responses-1.0.0"
)

DATASET_ROOT = (
    PROJECT_ROOT / "data" / "raw" / DATASET_NAME
)

DATA_DIR = DATASET_ROOT / "data"

print("Project root:", PROJECT_ROOT.resolve())
print("Dataset root:", DATASET_ROOT.resolve())
print("Data directory:", DATA_DIR.resolve())

assert DATA_DIR.exists(), f"Dataset folder not found: {DATA_DIR}"

print("CSV files:", [p.name for p in sorted(DATA_DIR.glob("*.csv"))])

Project root: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2
Dataset root: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\data\raw\qol_stress-a-multimodal-dataset-of-physiological-and-self-reported-stress-responses-1.0.0
Data directory: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\data\raw\qol_stress-a-multimodal-dataset-of-physiological-and-self-reported-stress-responses-1.0.0\data
CSV files: ['ecg.csv', 'eda.csv', 'fitbit_heart_rate_per_minute.csv', 'fitbit_heart_rate_per_second.csv', 'fitbit_heart_rate_summary.csv', 'participants.csv', 'pss_14.csv', 'stai.csv']


In [4]:
# 1. Confirm file inventory and sizes
import pandas as pd
files = sorted(DATASET_ROOT.rglob('*'))
inventory = pd.DataFrame([{'path': str(p.relative_to(DATASET_ROOT)), 'extension': p.suffix, 'bytes': p.stat().st_size}
                          for p in files if p.is_file()])
display(inventory)
inventory.to_csv(PROJECT_ROOT / 'reports' / 'tables' / 'raw_file_inventory.csv', index=False)

""


OSError: Cannot save file into a non-existent directory: 'd:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\notebooks\reports\tables'

In [ ]:
# 2. Load each CSV without changing the raw files
csv_paths = sorted(DATA_DIR.glob('*.csv'))
data = {p.stem: pd.read_csv(p, dtype={'username': 'string'}, low_memory=False) for p in csv_paths}
for name, df in data.items():
    print(f'\n{name}: {df.shape[0]:,} rows × {df.shape[1]} columns')
    display(pd.DataFrame({'column': df.columns, 'dtype': [str(df[c].dtype) for c in df.columns],
                          'missing': [int(df[c].isna().sum()) for c in df.columns],
                          'missing_pct': [round(float(df[c].isna().mean()*100), 2) for c in df.columns]}))

In [ ]:
# 3. Participant coverage by collection
id_candidates = ['username', 'participant_id', 'subject_id', 'participant']
coverage_rows = []
participant_sets = {}
for name, df in data.items():
    id_col = next((c for c in id_candidates if c in df.columns), None)
    if id_col:
        ids = set(df[id_col].dropna().astype(str).unique())
        participant_sets[name] = ids
        coverage_rows.append({'collection': name, 'rows': len(df), 'unique_participants': len(ids)})
coverage = pd.DataFrame(coverage_rows)
display(coverage)
coverage.to_csv(PROJECT_ROOT / 'reports' / 'tables' / 'participant_coverage.csv', index=False)
all_participants = sorted(set().union(*participant_sets.values())) if participant_sets else []
print('Unique participant IDs across collections:', len(all_participants))

In [ ]:
# 4. Phase label distribution; do not assume every collection has the same label structure
phase_rows = []
for name, df in data.items():
    if 'phase' in df.columns:
        counts = df['phase'].astype('string').str.lower().value_counts(dropna=False)
        for label, count in counts.items():
            phase_rows.append({'collection': name, 'phase': str(label), 'records': int(count)})
phase_distribution = pd.DataFrame(phase_rows)
display(phase_distribution)
phase_distribution.to_csv(PROJECT_ROOT / 'reports' / 'tables' / 'phase_distribution.csv', index=False)

In [ ]:
# 5. ECG and EDA structural checks
if 'ecg' in data:
    ecg = data['ecg']
    print('ECG columns:', ecg.columns.tolist())
    if 'waveform_samples' in ecg.columns:
        print('Waveform field examples (truncated):')
        display(ecg[['username', 'phase', 'waveform_samples']].head(2).assign(
            waveform_preview=lambda x: x['waveform_samples'].astype(str).str.slice(0, 180)
        )[['username', 'phase', 'waveform_preview']])
if 'eda' in data:
    eda = data['eda']
    print('EDA columns:', eda.columns.tolist())
    if 'time_sec' in eda.columns:
        t = pd.to_numeric(eda['time_sec'], errors='coerce')
        print('EDA time_sec rows:', len(t), 'unique:', t.nunique(), 'missing:', int(t.isna().sum()),
              'repeated markers:', int(t.duplicated().sum()))
    if 'valid_data' in eda.columns:
        display(eda['valid_data'].value_counts(dropna=False))

In [ ]:
# 6. Participant × phase coverage for ECG, EDA and STAI
for name in ['ecg', 'eda', 'stai']:
    if name in data and {'username', 'phase'}.issubset(data[name].columns):
        print(f'\n{name}:')
        display(pd.crosstab(data[name]['username'], data[name]['phase'], margins=True))

In [ ]:
# 7. Check duplicate rows and numeric ranges for review (not automatic deletion)
duplicate_summary = []
numeric_summaries = []
for name, df in data.items():
    duplicate_summary.append({'collection': name, 'duplicate_rows': int(df.duplicated().sum())})
    numeric = df.select_dtypes(include='number')
    if not numeric.empty:
        stats = numeric.describe().T.reset_index().rename(columns={'index': 'column'})
        stats.insert(0, 'collection', name)
        numeric_summaries.append(stats)
display(pd.DataFrame(duplicate_summary))
if numeric_summaries:
    numeric_report = pd.concat(numeric_summaries, ignore_index=True)
    display(numeric_report)
    numeric_report.to_csv(PROJECT_ROOT / 'reports' / 'tables' / 'numeric_summary.csv', index=False)

In [ ]:
# 8. Verify SHA256SUMS.txt without modifying any file
import hashlib, re
checksum_file = DATASET_ROOT / 'SHA256SUMS.txt'
checksum_results = []
if checksum_file.exists():
    for line in checksum_file.read_text(encoding='utf-8', errors='replace').splitlines():
        line = line.strip()
        if not line:
            continue
        match = re.match(r'^([A-Fa-f0-9]{64})\s+[* ]?(.*)$', line)
        if not match:
            checksum_results.append({'entry': line, 'status': 'UNPARSED_LINE'})
            continue
        expected, rel = match.groups()
        path = (DATASET_ROOT / rel).resolve()
        try:
            path.relative_to(DATASET_ROOT.resolve())
        except ValueError:
            checksum_results.append({'file': rel, 'status': 'PATH_OUTSIDE_DATASET'})
            continue
        if not path.is_file():
            checksum_results.append({'file': rel, 'status': 'MISSING_FILE'})
            continue
        h = hashlib.sha256(path.read_bytes()).hexdigest()
        checksum_results.append({'file': rel, 'status': 'OK' if h.lower() == expected.lower() else 'HASH_MISMATCH'})
else:
    print('SHA256SUMS.txt not found')
checksum_df = pd.DataFrame(checksum_results)
display(checksum_df)
checksum_df.to_csv(PROJECT_ROOT / 'reports' / 'tables' / 'checksum_verification.csv', index=False)

## Audit interpretation — complete after reviewing outputs

Do not begin model training until you have answered: (1) which modalities have both RELAX and STRESS data per participant; (2) what rows actually represent in each table; (3) how the phase label is assigned; (4) what missingness means; (5) which time series can legitimately be aligned; and (6) what is the independent evaluation unit.

For this dataset, subject-independent evaluation is essential: keep all records/windows from one participant together in a held-out fold. Do not use `username` as a model feature. Do not claim chronic astronaut stress detection from this controlled protocol.